In [ ]:
import json
import pandas as pd
from typing import Any, Optional, Dict

import requests

SERVICE_URL = 'https://api.fda.gov/drug/shortages.json'
TIMEOUT_SECONDS = 15

#The function for searching the API

#The function for searching the API
def search_for_drug_shortages(session: requests.Session, drug_name: str) -> Optional[Dict[str, Any]]:
    results = []
    searches = [
        f'generic_name:"{drug_name}"',
        f'proprietary_name:"{drug_name}"'
    ]

    for search in searches:
        params = {'search': search, 'limit': 100}

        try:
            response = session.get(SERVICE_URL, params=params, timeout=TIMEOUT_SECONDS)
            response.raise_for_status()
            data = response.json()
            results.extend(data.get('results', []))

        except requests.exceptions.HTTPError as err:
            if err.response is not None and err.response.status_code == 404:
                continue
            print('API response:', err.response.text if err.response is not None else err)
            return None

        except requests.exceptions.RequestException as err:
            print('Request error:', err)
            return None

    #Remove duplicate records
    unique_results = []
    seen = set()

    for drug in results:
        key = (drug.get('company_name'), drug.get('generic_name'),
               drug.get('proprietary_name'), drug.get('availability'))
        if key not in seen:
            seen.add(key)
            unique_results.append(drug)

    return {'results': unique_results}


#The function for printing results
def print_drug_shortages(shortages: Dict[str, Any]) -> None:
    table = []

    for drug in shortages.get('results', []):
        table.append({
            'Company Name': drug.get('company_name', 'N/A'),
            'Availability': drug.get('availability', 'N/A')
        })

    df = pd.DataFrame(table)

    if df.empty:
        print('No drug shortage records found.')
    else:
        print()
        
        display(df)
        print()
        
#The function for getting a user query
def main() -> None:
    with requests.Session() as session:
        while True:
            drug_name = input('Enter a drug name to search for shortages (blank to quit): ').strip()
            if not drug_name:
                break

            shortages = search_for_drug_shortages(session, drug_name)
            if not shortages:
                print('No drug shortage data returned.')
                continue

            print_drug_shortages(shortages)

if __name__ == '__main__':
    main()